# Lab 3 — Choose K

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Sweep candidate **k** values for KNN and record test accuracy for each.
2. Select the **best k** by highest validation accuracy.
3. Plot **accuracy vs k** to visualize the bias–variance tradeoff.
4. Relate small k (overfitting) and large k (underfitting) to neighbor voting.

> **Checkpoints:** best k = **3** · accuracy ≈ **0.59** · all seven k values evaluated

**Companion script:** `../scripts/lab03_choose_k.py`

## The bias–variance tradeoff in KNN

| k | Behavior | Risk |
|---|----------|------|
| **Small** (e.g. k=1) | Follows every local quirk in training data | **Overfitting** — high variance |
| **Large** (e.g. k=15) | Smooths votes across many distant neighbors | **Underfitting** — high bias |
| **Sweet spot** | Captures local structure without noise | Best test accuracy |

Lab 2 fixed k=5 (accuracy ≈ 0.55). This lab finds a better neighbor count on the **same** train/test split.

---

## 1. Load data and split (same as Lab 2)

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-04":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}, test size: {len(X_test)}")

---

## 2. Sweep k values

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15]
results: list[tuple[int, float]] = []

for k in k_values:
    pipe = Pipeline(
        steps=[
            ("scale", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k)),
        ]
    )
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results.append((k, acc))

best_k, best_acc = max(results, key=lambda item: item[1])

print("Lab 3 — Choose K")
print("k\taccuracy")
for k, acc in results:
    marker = " <-- best" if k == best_k else ""
    print(f"{k}\t{acc:.4f}{marker}")
print(f"best k: {best_k} (accuracy {best_acc:.4f})")

---

## 3. Results table

In [ ]:
results_df = pd.DataFrame(results, columns=["k", "accuracy"])
results_df["best"] = results_df["k"] == best_k
display(results_df.round(4))

k=**3** reaches ≈ **0.59** — matching Day 3 logistic regression on this sample. k=**11** ties on accuracy; `max()` picks the **first** best k in the sweep order (3 before 11).

---

## 4. Plot accuracy vs k

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=results_df, x="k", y="accuracy", marker="o", ax=ax, color="steelblue")
ax.axhline(best_acc, color="gray", linestyle="--", alpha=0.6, label=f"best acc = {best_acc:.2f}")
ax.scatter([best_k], [best_acc], color="crimson", s=120, zorder=5, label=f"best k = {best_k}")
ax.set_xlabel("k (neighbors)")
ax.set_ylabel("test accuracy")
ax.set_title("KNN: accuracy vs k")
ax.set_xticks(k_values)
ax.legend()
plt.tight_layout()
plt.show()

### Reading the curve

- **k=1** (0.57): very local — sensitive to noisy neighbors.
- **k=5** (0.55): Lab 2 default — not optimal here.
- **k=15** (0.57): smoother boundary — may miss local default patterns.
- **k=3** (0.59): best balance for this dataset and split.

---

## 5. Compare to Lab 2 (k=5)

In [ ]:
acc_k5 = next(acc for k, acc in results if k == 5)
improvement = best_acc - acc_k5

compare = pd.DataFrame({
    "setting": ["Lab 2 (k=5)", f"Lab 3 (k={best_k})"],
    "accuracy": [acc_k5, best_acc],
})
display(compare.round(4))
print(f"improvement over k=5: {improvement:+.4f}")

---

## 6. Checkpoint summary

In [ ]:
assert len(k_values) == 7
assert len(results) == 7
assert best_k == 3
assert abs(best_acc - 0.59) < 0.02
acc_by_k = dict(results)
assert abs(acc_by_k[5] - 0.55) < 0.02
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why is evaluating many k values on the **test** set a simplification? *(Production: use cross-validation or a held-out validation set.)*
2. What would you expect at k = len(X_train)?
3. Labs 4–6 reuse k=3 — why ship the tuned value in the API and MLflow runs?

**Previous:** [Lab 2 — KNN classifier](lab02_knn_classifier.ipynb)  
**Next:** [Lab 4 — FastAPI scoring API](lab04_fastapi_scoring_api.ipynb)